### ALL FUNCTIONS

In [ ]:

import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from matplotlib import pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from wordcloud import WordCloud
from glob import glob
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity

# Stopwords

def fetch_stopwords():
    filepath = "My_stopwords.txt"
    all_stopwords = set(stopwords.words('english'))
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            raw_data = f.read()
        data_fetch = raw_data.split(",")
        mine_stopwords = [word.strip().strip('"').strip("'").lower() for word in data_fetch if word.strip()]
        all_stopwords.update(mine_stopwords)
        return all_stopwords
    except FileNotFoundError:
        print("Le fichier My_stopwords.txt est introuvable. Les stopwords par défaut seront utilisés.")
        return all_stopwords
    except Exception as e:
        print(f"Une erreur est survenue : {e}")
        return all_stopwords


def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)


#Function to clean each book text
def preprocessing(book__path):

    try:
        with open(book__path, "r", encoding="utf-8", errors="ignore") as f:
            all_text = f.read()
    except FileNotFoundError:
        print(f"Fichier introuvable : {book__path}")
        return []

    # Supprimer le texte à la fin et retirer les lignes vides
    book_text = all_text.split("End of the Project")[0]
    lines = [line.strip() for line in book_text.split("\n") if line.strip()]

    # Tokenisation
    tokens = []
    for line in lines:
        tokens.extend(word_tokenize(line))

    mine_stopwords = fetch_stopwords()

    clean_tokens = [w for w in tokens if w.isalpha() and len(w) > 1]
    clean_tokens = [w.lower() for w in clean_tokens if w.lower() not in mine_stopwords]

    # Lemmatisation
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in clean_tokens]

    lemmatized_words = [w for w in lemmatized_words if w not in mine_stopwords]

    return lemmatized_words

# Function to get frequencies
def book_freq(cleaned_data):
   return nltk.FreqDist(cleaned_data)


#function to generate the word cloud
def books_cloud(frequency, book__name):
    #Importing WordCloud from wordcloud to create the words cloud
    book_cloud = WordCloud(background_color="white", width=1000, height=500, colormap='rainbow', ).generate_from_frequencies(frequency)
    plt.figure(figsize = (12, 12))
    plt.imshow(book_cloud)
    plt.title(book__name)
    plt.axis("off")
    plt.show()

#function to create the bag of word
def book_freq_data(frequence_):
    # book_df = pd.DataFrame.from_dict(frequence_, orient='index', columns=['Frequency'])
    book_df = pd.DataFrame(frequence_.items(), columns=['word', 'frequency'])
    # book_df.index.name = 'Word'
    book_df = book_df.sort_values(by='frequency', ascending=False)
    return book_df


In [ ]:
books_path = glob("../Data - NLP/*.txt")

bags_of_words = {}

for book_path in books_path:
    book_name = book_path.split("/")[-1].split(".")[0]
    data = preprocessing(book_path)
    frequence = book_freq(data)
    bag_of_word = book_freq_data(frequence).reset_index()
    bags = " ".join(bag_of_word['word'].tolist())
    bags_of_words[book_name] = bags


In [ ]:
# bags_of_words_df = pd.DataFrame(bags_of_words).reset_index(drop=True)
bags_of_words_df = pd.DataFrame.from_dict(bags_of_words, orient='index', columns=['Words_joined'])
bags_of_words_df = bags_of_words_df.sort_index(ascending=True)
bags_of_words_df = bags_of_words_df.reset_index()
bags_of_words_df.rename(columns={'index':'Books'}, inplace=True)
bags_of_words_df
# bags_of_words_df["Books"][0]


In [ ]:
vectorizer = TfidfVectorizer(max_df=0.8, min_df=2, ngram_range=(1, 2))

books_matrix = vectorizer.fit_transform(bags_of_words_df['Words_joined']).toarray()
books_matrix_df = pd.DataFrame(books_matrix)
books_matrix_df = books_matrix_df.transpose()

col_tab = []
for i in bags_of_words_df['Books'] :
    col_tab.append(i)
books_matrix_df.columns = col_tab
books_matrix_df = books_matrix_df.transpose()
# books_matrix

topics = vectorizer.get_feature_names_out()
# tempris
idx_tab = []
for i in topics:
    idx_tab.append(i)
#

books_matrix_df.columns = idx_tab
# books_matrix = books_matrix.transpose()
books_matrix_df
# print(books_matrix)

In [ ]:
count_vectorizer = CountVectorizer(max_df=0.8, min_df=2, ngram_range=(1, 2))

books_matrix_count = count_vectorizer.fit_transform(bags_of_words_df['Words_joined'])

books_matrix_count = pd.DataFrame(books_matrix_count.toarray())
books_matrix_count = books_matrix_count.transpose()
col_tab = []
for i in bags_of_words_df['Books'] :
    col_tab.append(i)
books_matrix_count.columns = col_tab

tempris = count_vectorizer.get_feature_names_out()
# tempris
idx_tab = []
for i in tempris:
    idx_tab.append(i)

books_matrix_count = books_matrix_count.transpose()
books_matrix_count.columns = idx_tab
# books_matrix_count = books_matrix_count.transpose()
books_matrix_count
# print(books_matrix_count)

In [ ]:
lsa_to = TruncatedSVD(n_components=3, random_state=42)
lsa_model = make_pipeline(lsa_to, Normalizer(copy=False))
# lsa_model = NMF(n_components=3, random_state=42)

lsa_books_matrix = lsa_model.fit_transform(books_matrix)

# print(lsa_books_matrix)
topic_cols = [f"topic{i+1}" for i in range(3)]
df_lsa_topics = pd.DataFrame(lsa_books_matrix, columns=topic_cols)
df_books_with_topics = pd.concat([bags_of_words_df, df_lsa_topics], axis=1)
df_books_with_topics

In [ ]:
terms = vectorizer.get_feature_names_out()
for i, component in enumerate(lsa_to.components_):
    top_ten_word_indices = component.argsort()[:-11:-1]
    top_words = [terms[idx] for idx in top_ten_word_indices]
    print(f"Topic {i+1}: {' , '.join(top_words)}")

# terms = vectorizer.get_feature_names_out()
# for i, component in enumerate(lsa_model.components_):
#     top_ten_word_indices = component.argsort()[:-11:-1]
#     top_words = [terms[idx] for idx in top_ten_word_indices]
#     print(f"Topic {i+1}: {' | '.join(top_words)}")



In [ ]:
df_book_without_terms = df_books_with_topics.copy()
df_book_without_terms.drop('Words_joined', axis=1, inplace=True)
df_book_without_terms = df_book_without_terms.set_index('Books')
# df_book_without_terms['topic1']
range_topic = []
topic_value = []
row_all = []
cpt = 0
for index, row in df_book_without_terms.iterrows():
    topic_max = row.idxmax()
    range_topic.append(topic_max)
    topic_value.append(float(row[topic_max]))

df_book_without_terms['main_topic'] = range_topic
df_book_without_terms['topic_value'] = topic_value
df_list = df_book_without_terms.reset_index()
# df_list

df_list_topic = df_list
df_list_topic1 = df_list_topic[df_list_topic['main_topic'] == 'topic1']
df_list_topic2 = df_list_topic[df_list_topic['main_topic'] == 'topic2']
df_list_topic3 = df_list_topic[df_list_topic['main_topic'] == 'topic3']
books_topic1 = []
books_topic2 = []
books_topic3 = []
topics_books = {}

for i in df_list_topic1['Books']:
    books_topic1.append(i)
for i in df_list_topic2['Books']:
    books_topic2.append(i)
for i in df_list_topic3['Books']:
    books_topic3.append(i)


topics_books['Topic 1'] = books_topic1
topics_books['Topic 2'] = books_topic2
topics_books['Topic 3'] = books_topic3

topics_books
# books_topic1

In [ ]:
def best_recommended_books(book_title, nbr):
    if book_title not in bags_of_words_df["Books"].values:
        print(f"Le livre '{book_title}' n'existe pas dans le corpus.")
        return None

    # Calcul similarity between all the books of the lsa_matrice
    cosine_sim = cosine_similarity(books_matrix)
    # print(f" Matrice du calcul des cosinus entre tous les livres : {cosine_sim}")

    book_index = bags_of_words_df[bags_of_words_df["Books"] == book_title].index[0]
    # print(f" Index du livre recherché : {book_index}")

    sim_scores = list(enumerate(cosine_sim[book_index]))
    # print(f"Matrice du calcul sous forme de liste pour le livre recherché et les autres livres: {sim_scores}")

    #trier en fonction de la valeur dans (40, np.float64(0.21374295934987794)), l'élément x[1] pour la valeur float
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Recuperer les valeurs pour lesquelles l'index n'est pas celui du livre recherché
    sim_scores = [s for s in sim_scores if s[0] != book_index]
    # print(f"List sans l'element recherché : {sim_scores}")

    top_similar_books = sim_scores[:nbr]
    # print(top_similar_books)
    recommendations = []
    for idx, score in top_similar_books:
        similar_book = bags_of_words_df.iloc[idx]["Books"]
        cos_value = round(score, 6)
        recommendations.append((similar_book, np.array([[cos_value]])))

    return recommendations


In [ ]:
best_recommended_books("alice-in-wonderland", 5)